In [1]:
import os
os.environ['CACHELEVEL'] = '0'

In [2]:
from tinygrad import Tensor
from tinygrad.shape.shapetracker import ShapeTracker
from tinygrad.shape.view import View, merge_dims, unravel
from tinygrad.ops import UOp

In [5]:
# dot product between [1,2] and [3,4] is 1 * 3 + 2 * 4 = 11
b = Tensor([1,2])


In [ ]:

b = Tensor([3,4])


In [ ]:

res = b.dot(b)
print(res.numpy()) # 11

In [7]:
t = Tensor(list(range(20)))
print(f"memory:\n {t.numpy()}\n")

memory:
 [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]



In [8]:
a = t[2:14].reshape(3, 4).pad((None, (0, 3)))#.shrink((None, (0, 4)))
print(f"tensor:\n {a.numpy()}\n")



tensor:
 [[ 2  3  4  5  0  0  0]
 [ 6  7  8  9  0  0  0]
 [10 11 12 13  0  0  0]]



In [9]:
a = t[2:14].reshape(3, 4).pad((None, (0, 3))).shrink(((1,3), (2, 7)))
print(f"tensor:\n {a.numpy()}\n")
print(f'{a.shape=}')
print(f'{a.lazydata.st.views=}')

tensor:
 [[ 8  9  0  0  0]
 [12 13  0  0  0]]

a.shape=(2, 5)
a.lazydata.st.views=(View(shape=(2, 5), strides=(4, 1), offset=8, mask=((0, 2), (0, 2)), contiguous=False),)


In [12]:
print(a.lazydata.st.views[0])
# print(a.lazydata.st.views[1])

v_1 = a.lazydata.st.views[0]
print(v_1)

a.reshape((1, 10))
print(a.lazydata.st.views)

# why no new view created? of view didn't update?

View(shape=(2, 5), strides=(4, 1), offset=8, mask=((0, 2), (0, 2)), contiguous=False)
View(shape=(2, 5), strides=(4, 1), offset=8, mask=((0, 2), (0, 2)), contiguous=False)
(View(shape=(2, 5), strides=(4, 1), offset=8, mask=((0, 2), (0, 2)), contiguous=False),)


In [13]:
print("--------------------B")
b = Tensor(list(range(20)))
print("b: ", b)
print("b.numpy():", b.numpy())
print("b.lazydata.st.views: ", b.lazydata.st.views)

--------------------B
b:  <Tensor <LB CUDA (20,) int (<Ops.COPY: 4>, None)> on CUDA with grad None>
b.numpy(): [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]
b.lazydata.st.views:  (View(shape=(20,), strides=(1,), offset=0, mask=None, contiguous=True),)


In [3]:
c = Tensor([[1,2,3], [4,5,6]])
print(c.numpy())
print(c.lazydata.st.views)


[[1 2 3]
 [4 5 6]]
(View(shape=(2, 3), strides=(3, 1), offset=0, mask=None, contiguous=True),)


In [7]:
d = c.permute((1, 0))
print(d.numpy())
print(d.lazydata.st.views)
print(c.transpose().numpy())


[[1 4]
 [2 5]
 [3 6]]
(View(shape=(3, 2), strides=(1, 3), offset=0, mask=None, contiguous=False),)
[[1 4]
 [2 5]
 [3 6]]


In [11]:
# Example of creating two incompatible views, by permuting and then reshaping

d = c.permute((1, 0)).reshape((1, 6))
print(d.shape)
print(d.lazydata.st.views)


(1, 6)
(View(shape=(3, 2), strides=(1, 3), offset=0, mask=None, contiguous=False), View(shape=(1, 6), strides=(0, 1), offset=0, mask=None, contiguous=True))


In [14]:
print("--------------------C")
c = Tensor.eye(3).realize()
print(c.numpy())

# assert not a.lazydata.is_unrealized_const()

# print("b.realize: ", b.realize())

# print("b.lazydata.base: ", b.lazydata.base)
print()

print(f'{c.shape=}')
print(f'{c.lazydata.st.views=}')
# idx, valid = c.lazydata.st.expr_idxs()
# print(f'{idx=}')  # index -> buf location
# print(f'{valid=}')  # mask

print(c.lazydata.st.views[0])
print(c.lazydata.st.views[1])

# why are there two views here?

--------------------C
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

c.shape=(3, 3)
c.lazydata.st.views=(View(shape=(3, 4), strides=(0, 0), offset=0, mask=((0, 3), (0, 1)), contiguous=False), View(shape=(3, 3), strides=(3, 1), offset=0, mask=None, contiguous=True))
View(shape=(3, 4), strides=(0, 0), offset=0, mask=((0, 3), (0, 1)), contiguous=False)
View(shape=(3, 3), strides=(3, 1), offset=0, mask=None, contiguous=True)


In [16]:
print("--------------------C")

s = ShapeTracker(views=(
    # idk random
    View.create(shape=(20,), strides=(1,), offset=0, mask=None),
    View.create(shape=(3, 5), strides=(3, 1), offset=2, mask=((0, 3), (0, 3))),
))
print(s)
print(s.simplify())

v = View.create(shape=(9,), strides=(1,), offset=0, mask=None)
print(v.reshape((3, 3)))

--------------------C
ShapeTracker(views=(View(shape=(20,), strides=(1,), offset=0, mask=None, contiguous=True), View(shape=(3, 5), strides=(3, 1), offset=2, mask=((0, 3), (0, 3)), contiguous=False)))
ShapeTracker(views=(View(shape=(3, 5), strides=(3, 1), offset=2, mask=((0, 3), (0, 3)), contiguous=False),))
View(shape=(3, 3), strides=(3, 1), offset=0, mask=None, contiguous=True)


In [ ]:
# found a 3 view shapetracker that can be expressed with 2 views.

View(shape=(1, 1, 2, 4, 2, 4), strides=(0, 0, 2, 8, 1, 4), offset=0, mask=((0, 1), (0, 1), (0, 2), (0, 2), (0, 2), (0, 2)), contiguous=False)
View(shape=(1, 1, 9, 9), strides=(0, 0, 8, 1), offset=0, mask=((0, 1), (0, 1), (0, 8), (0, 8)), contiguous=False)
View(shape=(1, 1, 1, 1, 3, 3, 3, 3), strides=(0, 0, 0, 0, 27, 9, 3, 1), offset=0, mask=None, contiguous=True)


# the trick is that you can have extra mask on the final view because it's already masked on the top one

View(shape=(1, 1, 2, 4, 2, 4), strides=(0, 0, 2, 8, 1, 4), offset=0, mask=((0, 1), (0, 1), (0, 2), (0, 2), (0, 2), (0, 2)), contiguous=False)
View(shape=(1, 1, 1, 1, 3, 3, 3, 3), strides=(0, 0, 0, 0, 24, 8, 3, 1), offset=0, mask=((0, 1), (0, 1), (0, 1), (0, 1), (0, 2), (0, 3), (0, 2), (0, 3)), contiguous=False)

In [17]:
memory = Tensor.arange(start=0, stop=64, step=1)
print(f"memory:\n{memory.numpy()}\n")


t1 = Tensor.arange(start=0, stop=64, step=1)
t1_shaped = t1.reshape((1, 1, 2, 4, 2, 4))
print(f"t1_shaped:\n{t1_shaped.numpy()}\n")
print(f"t1_shaped.shapetracker: {t1_shaped.lazydata.st} \n")


t1.lazydata.st.stride((1, 1, 2, 8, 1, 4))
print(f"t1_shaped.shapetracker: {t1_shaped.lazydata.st}\n")

# how can I change the strides to match the views above? .stride() does not allow 0 in any stride dimension for some reason.

# print(f"t1:\n{t1.realize().numpy()}\n")



# t2 = Tensor.arange(start=0, stop=81, step=1)
# t2_shaped = t2.reshape((1, 1, 9, 9))
# print(f"t2_shaped:\n{t2_shaped.numpy()}\n")

# t3 = Tensor.arange(start=0, stop=81, step=1)
# t3_shaped = t3.reshape((1, 1, 1, 1, 3, 3, 3, 3))
# print(f"t3_shaped:\n{t3_shaped.numpy()}\n")

memory:
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63]

t1_shaped:
[[[[[[ 0  1  2  3]
     [ 4  5  6  7]]

    [[ 8  9 10 11]
     [12 13 14 15]]

    [[16 17 18 19]
     [20 21 22 23]]

    [[24 25 26 27]
     [28 29 30 31]]]


   [[[32 33 34 35]
     [36 37 38 39]]

    [[40 41 42 43]
     [44 45 46 47]]

    [[48 49 50 51]
     [52 53 54 55]]

    [[56 57 58 59]
     [60 61 62 63]]]]]]

t1_shaped.shapetracker: ShapeTracker(views=(View(shape=(1, 1, 2, 4, 2, 4), strides=(0, 0, 32, 8, 4, 1), offset=0, mask=None, contiguous=True),)) 

t1_shaped.shapetracker: ShapeTracker(views=(View(shape=(1, 1, 2, 4, 2, 4), strides=(0, 0, 32, 8, 4, 1), offset=0, mask=None, contiguous=True),))



In [12]:
from tinygrad.shape.view import _merge_dims

# Example of dimension merging, where we need at least 2 dimensions
# 
# The reason (IMO) is that one dimension contains the actual data,
# and the other dimension is basically only there to say how
# many copies of that data we want.

shape = (2,2,2)
strides = (0,0,1)
print(_merge_dims(shape, strides))
# => ((4, 0, 0), (2, 1, 2))
#
# Meaning: 2 dimensions
#   (4, 0, 0): 4 is size of dimension, 0 stride, 0 underlying data
#   (2, 1, 2): 2 is size of dimension, 1 stride, 2 underlying data

((4, 0, 0), (2, 1, 2))


In [9]:
x = Tensor.arange(0, 6).reshape((2,3))
idx, valid = x.pad(((1, 1), (0, 0))).lazydata.st.to_indexed_uops()
idx.render(), valid.render()

('(((ridx0*3)+ridx1)+-3)', '((ridx0<3)&((ridx0<1)!=True))')

In [10]:
assert len(View.create((10,10)).minify().shape) == 1
assert len(View.create((10,10)).permute((1,0)).minify().shape) == 2
assert len(View.create((10,10,10,10)).permute((1,0,2,3)).minify().shape) == 3

In [14]:
View.create((10,10,10,10)).permute((1,0,2,3)).minify().shape

(10, 10, 100)

In [15]:
View.create((8, 6, 11), (66, 11, 1), 0, None)

View(shape=(8, 6, 11), strides=(66, 11, 1), offset=0, mask=None, contiguous=True)

In [17]:
# Example of a view that can't be simplified
ShapeTracker((
      View.create((8, 3, 1, 2, 11, 1), (33, 11, 0, 0, 1, 0), 0, None),
      View.create((8, 6, 11), (66, 11, 1), 0, None))).simplify()

ShapeTracker(views=(View(shape=(8, 3, 1, 2, 11, 1), strides=(33, 11, 0, 0, 1, 0), offset=0, mask=None, contiguous=False), View(shape=(8, 6, 11), strides=(66, 11, 1), offset=0, mask=None, contiguous=True)))

In [22]:
# Example of a view that can be simplified
ShapeTracker((
      View.create((1, 3, 2, 11, 4, 28),       (0, 308, 0, 28, 0, 1), 0, None),
      View.create((1, 3, 2, 11, 26, 1, 1, 3), (0, 2464, 0, 112, 1, 0, 0, 29), 0, None))).simplify().views


(View(shape=(1, 3, 2, 11, 26, 1, 1, 3), strides=(0, 308, 0, 28, 1, 0, 0, 1), offset=0, mask=None, contiguous=False),)

In [24]:
# Simple example of a view that can be simplified
ShapeTracker((
      View.create((4,2,4), (8,4,1), 0, None),
      View.create((8,4), (4,1), 0, None)
)).simplify().views


(View(shape=(8, 4), strides=(4, 1), offset=0, mask=None, contiguous=True),)

In [ ]:
# Simple example of a view that can not be simplified
ShapeTracker((
      View.create((3, 2), (1, 3), 0, None),  # this is the permute
      View.create((1,6), (0,1), 0, None)     # this is the reshape
)).simplify().views

# This is how this ShapeTracker could be created:
# ShapeTracker((View.create((2,3), (3,1), 0, None), )).permute((1,0)).reshape((1,6)).simplify()

(View(shape=(8, 4), strides=(4, 1), offset=0, mask=None, contiguous=True),)

In [ ]:
ShapeTracker((View.create((2,3), (3,1), 0, None), )).permute((1,0)).reshape((1,6)).simplify()

In [8]:
ShapeTracker((View.create((2,3), (3,1), 0, None), )).permute((1,0)).reshape((1,6)).simplify()


ShapeTracker(views=(View(shape=(3, 2), strides=(1, 3), offset=0, mask=None, contiguous=False), View(shape=(1, 6), strides=(0, 1), offset=0, mask=None, contiguous=True)))

In [16]:
orig = Tensor([[1,2,3], [4,5,6]])
mixed_up = orig.permute((1,0)).reshape((1,6))
print(orig.numpy())
print(orig.lazydata.st.views)
print(mixed_up.numpy())
print(mixed_up.lazydata.st.views)

mixed_up.realize()
print(mixed_up.lazydata.st.views)




[[1 2 3]
 [4 5 6]]
(View(shape=(2, 3), strides=(3, 1), offset=0, mask=None, contiguous=True),)
[[1 4 2 5 3 6]]
(View(shape=(3, 2), strides=(1, 3), offset=0, mask=None, contiguous=False), View(shape=(1, 6), strides=(0, 1), offset=0, mask=None, contiguous=True))
(View(shape=(3, 2), strides=(1, 3), offset=0, mask=None, contiguous=False), View(shape=(1, 6), strides=(0, 1), offset=0, mask=None, contiguous=True))


In [18]:
idx, valid = mixed_up.lazydata.st.to_indexed_uops()
print(idx.render())
print(valid.render())


(((ridx1%2)*3)+(ridx1//2))
True


In [13]:
orig.lazydata.base is mixed_up.lazydata.base

True

In [14]:
mixed_up.numpy()

array([[1, 4, 2, 5, 3, 6]], dtype=int32)

In [3]:
orig = Tensor([[1,2,3], [4,5,6]])
mixed_up = orig.permute((1,0)).reshape((1,6))
mixed_up.lazydata.st.simplify()

ShapeTracker(views=(View(shape=(3, 2), strides=(1, 3), offset=0, mask=None, contiguous=False), View(shape=(1, 6), strides=(0, 1), offset=0, mask=None, contiguous=True)))

In [4]:
mask_example = Tensor.arange(6).reshape((2,3)).pad(((1,1), (0, 2))).shrink(((0,3), (2, 4)))
print(mask_example.lazydata.st.views)
print(mask_example.numpy())


(View(shape=(3, 2), strides=(3, 0), offset=-1, mask=((1, 3), (0, 1)), contiguous=False),)
[[0 0]
 [2 0]
 [5 0]]


In [2]:
mask_example = Tensor.arange(6).reshape((2,3)).pad(((1,1), None))
print(mask_example.lazydata.st.views)
print(mask_example.numpy())


(View(shape=(4, 3), strides=(3, 1), offset=-3, mask=((1, 3), (0, 3)), contiguous=False),)
[[0 0 0]
 [0 1 2]
 [3 4 5]
 [0 0 0]]


In [15]:
diag = Tensor.eye(3).realize()
print(diag.lazydata.st.views)
print(diag.numpy())
print(diag.lazydata.buffer)



(View(shape=(3, 4), strides=(0, 0), offset=0, mask=((0, 3), (0, 1)), contiguous=False), View(shape=(3, 3), strides=(3, 1), offset=0, mask=None, contiguous=True))
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


AttributeError: 'LazyBuffer' object has no attribute 'buffer'

In [20]:
ShapeTracker((
      View.create((4,4,2), (1,8,4), 0, None),
      View.create((8,4), (4,1), 0, None)
)).simplify().views

(View(shape=(4, 4, 2), strides=(1, 8, 4), offset=0, mask=None, contiguous=False),
 View(shape=(8, 4), strides=(4, 1), offset=0, mask=None, contiguous=True))

In [22]:
# Complicated example that simplifies

ShapeTracker((
      View.create((1, 3, 2, 11, 4, 28), (0, 308, 0, 28, 0, 1), 0, None),
      View.create((1, 3, 2, 11, 26, 1, 1, 3), (0, 2464, 0, 112, 1, 0, 0, 29), 0, None))).views

(View(shape=(1, 3, 2, 11, 4, 28), strides=(0, 308, 0, 28, 0, 1), offset=0, mask=None, contiguous=False),
 View(shape=(1, 3, 2, 11, 26, 1, 1, 3), strides=(0, 2464, 0, 112, 1, 0, 0, 29), offset=0, mask=None, contiguous=False))

In [24]:
# Complicated example that simplifies

ShapeTracker((
      #            x     x      x
      View.create((1, 3, 2, 11, 4, 28),       (0, 308,  0, 28, 0, 1), 0, None),
      View.create((1, 3, 2, 11,    26, 1, 1, 3), (0, 2464, 0, 112,   1, 0, 0, 29), 0, None))).simplify().views
      #            x     x             x  x

(View(shape=(1, 3, 2, 11, 26, 1, 1, 3), strides=(0, 308, 0, 28, 1, 0, 0, 1), offset=0, mask=None, contiguous=False),)

In [25]:
# Complicated example that simplifies

ShapeTracker((
      #                    
      View.create(( 3, 11, 28),    (308,  28,  1), 0, None),
      View.create(( 3, 11, 26, 3), (2464, 112, 1, 29), 0, None))).simplify().views
      #                        

(View(shape=(3, 11, 26, 3), strides=(2464, 112, 1, 29), offset=0, mask=None, contiguous=False),)

In [28]:
ShapeTracker((
      View.create((3, 6),    (12, 2), 0, None),
      View.create((3, 3), (6, 2), 0, None))).simplify().views

(View(shape=(3, 3), strides=(12, 4), offset=0, mask=None, contiguous=False),)

In [26]:
112 * 29

3248

In [3]:
ShapeTracker((
      View.create((112, 10), (11, 7), 0, None),
      View.create((9, ), (11, ), 0, None))).simplify().views

(View(shape=(9,), strides=(18,), offset=0, mask=None, contiguous=False),)

In [2]:
ShapeTracker((
      View.create((11, 4), (11, 7), 0, None),
      View.create((2, ), (7, ), 0, None))).simplify().views

(View(shape=(2,), strides=(32,), offset=0, mask=None, contiguous=False),)

In [2]:
ShapeTracker((
      View.create((11, 4), (11, 7), 0, None),
      View.create((3, ), (7, ), 0, None))).simplify().views

(View(shape=(11, 4), strides=(11, 7), offset=0, mask=None, contiguous=False),
 View(shape=(3,), strides=(7,), offset=0, mask=None, contiguous=False))

In [3]:
ShapeTracker((
      View.create((11, 4), (11, 7), 0, None),
      View.create((4, 3, ), (6, 7, ), 0, None))).simplify().views

(View(shape=(11, 4), strides=(11, 7), offset=0, mask=None, contiguous=False),
 View(shape=(4, 3), strides=(6, 7), offset=0, mask=None, contiguous=False))

In [8]:
ShapeTracker((
      View.create((4, 4), (28, 7), 0, None),
      View.create((4, ), (7, ), 0, None))).simplify().views

(View(shape=(4, 4), strides=(28, 7), offset=0, mask=None, contiguous=False),
 View(shape=(4,), strides=(7,), offset=0, mask=None, contiguous=False))

In [15]:
ShapeTracker((
      View.create((4, 4, 4), (28, 7, 1), 0, None),
      View.create((4, ), (7 * 4 + 1, ), 0, None))).simplify().views

(View(shape=(4, 4, 4), strides=(28, 7, 1), offset=0, mask=None, contiguous=False),
 View(shape=(4,), strides=(29,), offset=0, mask=None, contiguous=False))

In [18]:
ShapeTracker((
      View.create((4, 4), (11, 423), 0, None),
      View.create((4, ), (5, ), 0, None))).simplify().views

(View(shape=(4,), strides=(434,), offset=0, mask=None, contiguous=False),)

In [6]:
ShapeTracker((
      View.create((10,9,4), (8 * 11 + 4 * 13, 11, 13), 0, None),
      View.create((6, ), (9, ), 0, None))).simplify().views

(View(shape=(10, 9, 4), strides=(140, 11, 13), offset=0, mask=None, contiguous=False),
 View(shape=(6,), strides=(9,), offset=0, mask=None, contiguous=False))

In [8]:
terms = [unravel((2, 9, 4), 9*i) for i in range(6)]
[(term, term[0] * 140 + term[1] * 11 + term[2] * 13) for term in terms] 


[([0, 0, 0], 0),
 ([0, 2, 1], 35),
 ([0, 4, 2], 70),
 ([0, 6, 3], 105),
 ([1, 0, 0], 140),
 ([1, 2, 1], 175)]

In [42]:
a = Tensor(list(range(360)))
print(f"{a.numpy()=}")

a.numpy()=array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177, 17

In [43]:
a.reshape(10, 9, 4).numpy()


array([[[  0,   1,   2,   3],
        [  4,   5,   6,   7],
        [  8,   9,  10,  11],
        [ 12,  13,  14,  15],
        [ 16,  17,  18,  19],
        [ 20,  21,  22,  23],
        [ 24,  25,  26,  27],
        [ 28,  29,  30,  31],
        [ 32,  33,  34,  35]],

       [[ 36,  37,  38,  39],
        [ 40,  41,  42,  43],
        [ 44,  45,  46,  47],
        [ 48,  49,  50,  51],
        [ 52,  53,  54,  55],
        [ 56,  57,  58,  59],
        [ 60,  61,  62,  63],
        [ 64,  65,  66,  67],
        [ 68,  69,  70,  71]],

       [[ 72,  73,  74,  75],
        [ 76,  77,  78,  79],
        [ 80,  81,  82,  83],
        [ 84,  85,  86,  87],
        [ 88,  89,  90,  91],
        [ 92,  93,  94,  95],
        [ 96,  97,  98,  99],
        [100, 101, 102, 103],
        [104, 105, 106, 107]],

       [[108, 109, 110, 111],
        [112, 113, 114, 115],
        [116, 117, 118, 119],
        [120, 121, 122, 123],
        [124, 125, 126, 127],
        [128, 129, 130, 131],
    

In [52]:
st1 = ShapeTracker((View.create((10, 9, 4), (8 * 11 + 4 * 13, 11, 13), 0, None), View.create((6,), (9,), 0, None)))
st_simplfied = ShapeTracker((View.create((6,), (2 * 11 + 13,), 0, None), ))

x = a.reshape(10, 9, 4)
x.lazydata.st = st1

y = a.reshape(6, 60)
y.lazydata.st = st_simplfied

x.numpy(), y.numpy()


(array([  0,  35,  70, 105, 140, 175], dtype=int32),
 array([  0,  35,  70, 105, 140, 175], dtype=int32))

In [53]:
8 * 11 + 4 * 13, 2 * 11 + 

140

In [ ]:
st1 = ShapeTracker((View.create((10, 9, 4), (8 * 11 + 4 * 13, 11, 13), 0, None), View.create((6,), (9,), 0, None)))
st_simplfied = ShapeTracker((View.create((6,), (2 * 11 + 13,), 0, None), ))

x = a.reshape(10, 9, 4)
x.lazydata.st = st1

y = a.reshape(6, 60)
y.lazydata.st = st_simplfied

x.numpy(), y.numpy()


(array([  0,  35,  70, 105, 140, 175], dtype=int32),
 array([  0,  35,  70, 105, 140, 175], dtype=int32))

In [56]:
idx1, valid1 = st1.to_indexed_uops()
idx_simplified, valid_simplified = st_simplfied.to_indexed_uops()

idx1.render(), valid1.render(), idx_simplified.render(), valid_simplified.render()


('((((ridx0//4)*140)+((((ridx0*9)//4)%9)*11))+((ridx0%4)*13))',
 'True',
 '(ridx0*35)',
 'True')

In [27]:
a.numpy()

AttributeError: 'tuple' object has no attribute 'shape'

In [14]:
xlazy = UOp(
  # device=a.lazydata.device,
  st=ShapeTracker((View.create((10, 9, 4), (8 * 11 + 4 * 13, 11, 13), 0, None), View.create((6,), (9,), 0, None))),
  dtype=a.lazydata.dtype,
  base=a.lazydata.base,
)
x = Tensor(xlazy)
print(x.numpy())
ylazy = UOp(
  # device=a.lazydata.device,
  st=ShapeTracker((View.create((6,), (2 * 11 + 13,), 0, None), )),
  dtype=a.lazydata.dtype,
  base=a.lazydata.base,
)
y = Tensor(ylazy)
print(y.numpy())

TypeError: UOpMetaClass.__call__() got an unexpected keyword argument 'st'

In [12]:
v0 = View.create((10, 9, 4), (8 * 11 + 4 * 13, 11, 13), 0, None)
v1 = View.create((6,), (9,), 0, None)

v = v0 + v1

print(f"{v=}")

v=None


In [5]:
View.create((4, 4), (28, 7), 0, None).reshape((16,))

View(shape=(16,), strides=(7,), offset=0, mask=None, contiguous=False)